In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/cdl"

pixels = spark.read.parquet(f"{BASE}/pixels/pixels")
names = (spark.read.csv(f"{BASE}/crop_names.csv", header=True, inferSchema=True)
         .select(F.col("Value").alias("crop_code"), "Class_Name")
         .filter(F.col("Class_Name").isNotNull()))

pixels.printSchema()
names.show(5)
print(pixels.count())

root
 |-- x: integer (nullable = true)
 |-- y: integer (nullable = true)
 |-- crop_code: short (nullable = true)

+---------+----------+
|crop_code|Class_Name|
+---------+----------+
|        0|Background|
|        1|      Corn|
|        2|    Cotton|
|        3|      Rice|
|        4|   Sorghum|
+---------+----------+
only showing top 5 rows
561008172


In [0]:
STRIDE = 4                       # must match the conversion script
PIXEL_HA = STRIDE**2 * 0.09      # each kept pixel = 16 original 30 m pixels = 1.44 ha
spark.conf.set("spark.sql.shuffle.partitions", "64")

GRAIN   = [1,3,4,12,13,21,22,23,24,25,27,28,29,30,39,205]
OILSEED = [5,6,10,31,32,33,34,35,38]
HAY     = [36,37,58,59,60]

crops = (pixels
    # Filter 1 (early): cropland codes only
    .filter(F.col("crop_code").between(1, 61) |
            F.col("crop_code").between(66, 77) |
            F.col("crop_code").between(204, 254))
    # Filter 2 (early): drop Fallow/Idle (61) -> actively planted land only
    .filter(F.col("crop_code") != 61)
    # Column transformations
    .withColumn("grid_x", F.floor(F.col("x") / 50000).cast("int"))   # 50 km grid cells
    .withColumn("grid_y", F.floor(F.col("y") / 50000).cast("int"))
    .withColumn("area_ha", F.lit(PIXEL_HA))
    .withColumn("land_group",
        F.when(F.col("crop_code").isin(GRAIN), "grain")
         .when(F.col("crop_code").isin(OILSEED), "oilseed")
         .when(F.col("crop_code").isin(HAY), "hay_forage")
         .when(F.col("crop_code").between(225, 241) | F.col("crop_code").isin(26, 254), "double_crop")
         .otherwise("specialty_other")))

# Join: broadcast the small lookup table, so the big table isn't shuffled
enriched = crops.join(F.broadcast(names), "crop_code")
enriched.show(5)

+---------+--------+-------+------+------+-------+----------+--------------------+
|crop_code|       x|      y|grid_x|grid_y|area_ha|land_group|          Class_Name|
+---------+--------+-------+------+------+-------+----------+--------------------+
|       36|-2149500|2399610|   -43|    47|   1.44|hay_forage|             Alfalfa|
|       36|-2149140|2399610|   -43|    47|   1.44|hay_forage|             Alfalfa|
|       36|-2145660|2399610|   -43|    47|   1.44|hay_forage|             Alfalfa|
|       37|-2144460|2399610|   -43|    47|   1.44|hay_forage|Other Hay/Non Alf...|
|       36|-2114100|2399610|   -43|    47|   1.44|hay_forage|             Alfalfa|
+---------+--------+-------+------+------+-------+----------+--------------------+
only showing top 5 rows


In [0]:
crop_totals = (enriched.groupBy("crop_code", "Class_Name", "land_group")
    .agg(F.sum("area_ha").alias("area_ha"), F.count("*").alias("pixels"))
    .orderBy(F.desc("area_ha")))

grid_stats = (enriched.groupBy("grid_x", "grid_y")
    .agg(F.sum("area_ha").alias("crop_ha"),
         F.countDistinct("crop_code").alias("n_crops"),
         F.sum(F.when(F.col("crop_code") == 1, F.col("area_ha")).otherwise(0)).alias("corn_ha"),
         F.sum(F.when(F.col("crop_code") == 5, F.col("area_ha")).otherwise(0)).alias("soy_ha"))
    .withColumn("corn_soy_share",
                F.round((F.col("corn_ha") + F.col("soy_ha")) / F.col("crop_ha"), 3)))

display(crop_totals.limit(10))
display(grid_stats.orderBy(F.desc("crop_ha")).limit(10))

crop_code,Class_Name,land_group,area_ha,pixels
1,Corn,grain,3.928556160129416E7,27281640
5,Soybeans,oilseed,3.1169750399699885E7,21645660
24,Winter Wheat,grain,1.0012085279880168E7,6952837
36,Alfalfa,hay_forage,5342096.159989684,3709789
37,Other Hay/Non Alfalfa,hay_forage,4334168.160006083,3009839
23,Spring Wheat,grain,3957481.439958713,2748251
2,Cotton,specialty_other,3727350.7199675557,2588438
4,Sorghum,grain,3333281.7599998326,2314779
26,Dbl Crop WinWht/Soybeans,double_crop,1505859.8400017405,1045736
3,Rice,grain,1097723.5200010573,762308


grid_x,grid_y,crop_ha,n_crops,corn_ha,soy_ha,corn_soy_share
12,39,221829.1200000886,21,111408.48000004815,106633.44000004044,0.983
-2,54,217447.200000058,25,46103.04000000856,69896.16000004702,0.533
12,40,216918.72000007937,15,109808.64000004556,102517.92000003379,0.979
13,39,216280.8000000773,17,110315.52000004638,100683.36000003082,0.976
3,44,215117.28000006804,15,127226.88000005766,84870.72000001033,0.986
1,43,213144.48000017725,16,114841.44000010396,93712.32000007316,0.978
-2,56,212322.24000008998,21,13947.840000001033,66879.36000004214,0.381
0,45,211070.88000006686,21,114806.88000005366,89680.32000001303,0.969
0,44,210247.20000005545,17,116310.2400000412,89028.00000001417,0.977
-3,39,210153.6000000833,17,144564.48000010176,60600.95999998152,0.976


In [0]:
enriched.createOrReplaceTempView("cdl")

In [0]:
%sql
SELECT Class_Name,
       land_group,
       ROUND(SUM(area_ha) / 1e6, 2) AS million_ha
FROM cdl
GROUP BY Class_Name, land_group
ORDER BY million_ha DESC
LIMIT 10

Class_Name,land_group,million_ha
Corn,grain,39.29
Soybeans,oilseed,31.17
Winter Wheat,grain,10.01
Alfalfa,hay_forage,5.34
Other Hay/Non Alfalfa,hay_forage,4.33
Spring Wheat,grain,3.96
Cotton,specialty_other,3.73
Sorghum,grain,3.33
Dbl Crop WinWht/Soybeans,double_crop,1.51
Rice,grain,1.1


In [0]:
%sql
WITH cell_crop AS (
  SELECT grid_x, grid_y, Class_Name, SUM(area_ha) AS ha
  FROM cdl
  GROUP BY grid_x, grid_y, Class_Name
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY grid_x, grid_y ORDER BY ha DESC) AS rn,
         SUM(ha)      OVER (PARTITION BY grid_x, grid_y)                 AS cell_ha
  FROM cell_crop
)
SELECT Class_Name            AS dominant_crop,
       COUNT(*)              AS n_cells,
       ROUND(AVG(ha / cell_ha), 3) AS avg_share_of_cell
FROM ranked
WHERE rn = 1 AND cell_ha >= 10000
GROUP BY Class_Name
ORDER BY n_cells DESC

dominant_crop,n_cells,avg_share_of_cell
Corn,591,0.461
Soybeans,310,0.452
Winter Wheat,221,0.464
Alfalfa,125,0.413
Spring Wheat,62,0.331
Cotton,57,0.486
Other Hay/Non Alfalfa,38,0.365
Peanuts,21,0.311
Sorghum,14,0.425
Almonds,14,0.361


In [0]:
%sql
SELECT grid_x, grid_y,
       ROUND(SUM(area_ha)) AS crop_ha,
       ROUND(SUM(CASE WHEN Class_Name IN ('Corn','Soybeans') THEN area_ha ELSE 0 END) / SUM(area_ha), 3) AS corn_soy_share,
       COUNT(DISTINCT Class_Name) AS n_crops
FROM cdl
GROUP BY grid_x, grid_y
HAVING SUM(area_ha) >= 10000
ORDER BY corn_soy_share DESC
LIMIT 15

grid_x,grid_y,crop_ha,corn_soy_share,n_crops
3,43,203668.0,0.987,17
2,43,208945.0,0.987,19
3,44,215117.0,0.986,15
13,37,203252.0,0.986,22
0,39,174588.0,0.985,16
4,44,195523.0,0.983,14
12,39,221829.0,0.983,21
13,38,204163.0,0.981,19
11,38,208745.0,0.981,14
12,38,208636.0,0.981,18


In [0]:
OUT = f"{BASE}/output"

# Partitioned write: one folder per land_group
(enriched.select("x", "y", "crop_code", "Class_Name", "land_group", "grid_x", "grid_y", "area_ha")
    .write.mode("overwrite").partitionBy("land_group")
    .parquet(f"{OUT}/crops_by_group"))

crop_totals.write.mode("overwrite").parquet(f"{OUT}/crop_totals")
grid_stats.write.mode("overwrite").parquet(f"{OUT}/grid_stats")
print("Write complete")

Write complete


In [0]:
grain_only = spark.read.parquet(f"{OUT}/crops_by_group").filter("land_group = 'grain'")
grain_only.explain(mode="formatted")   # look for PartitionFilters: only the grain folder is read
print(grain_only.count())

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [8]: [x#11316, y#11317, crop_code#11318, Class_Name#11319, grid_x#11320, grid_y#11321, area_ha#11322, land_group#11323]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/cdl/output/crops_by_group]
PartitionFilters: [isnotnull(land_group#11323), (land_group#11323 = grain)]
ReadSchema: struct<x:int,y:int,crop_code:smallint,Class_Name:string,grid_x:int,grid_y:int,area_ha:double>

(2) PhotonColumnarToRow
Input [8]: [x#11316, y#11317, crop_code#11318, Class_Name#11319, grid_x#11320, grid_y#11321, area_ha#11322, land_group#11323]

(3) PhotonResultStage
Input [8]: [x#11316, y#11317, crop_code#11318, Class_Name#11319, grid_x#11320, grid_y#11321, area_ha#11322, land_group#11323]


== Photon Explanation ==
The query is fully supported by Photon.
43297324


In [0]:
grid_stats.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   PhotonResultStage (24)
   +- PhotonColumnarToRow (23)
      +- PhotonProject (22)
         +- PhotonGroupingAgg (21)
            +- PhotonShuffleExchangeSource (20)
               +- PhotonShuffleMapStage (19)
                  +- PhotonShuffleExchangeSink (18)
                     +- PhotonGroupingAgg (17)
                        +- PhotonGroupingAgg (16)
                           +- PhotonShuffleExchangeSource (15)
                              +- PhotonShuffleMapStage (14)
                                 +- PhotonShuffleExchangeSink (13)
                                    +- PhotonGroupingAgg (12)
                                       +- PhotonProject (11)
                                          +- PhotonBroadcastHashJoin Inner (10)
                                             :- PhotonProject (2)
                                             :  +- PhotonScan parquet  (1)
                                       

In [0]:
import time
def time_it(label, fn):
    t = time.perf_counter(); r = fn()
    print(f"{label}: {time.perf_counter() - t:.2f}s"); return r

# "Bad" order: join and add columns first, filter last, no broadcast hint
bad = (pixels.join(names, "crop_code")
       .withColumn("area_ha", F.lit(PIXEL_HA))
       .filter(F.col("crop_code") == 1)
       .groupBy("Class_Name").agg(F.sum("area_ha")))

# Optimized: filter first, keep only needed columns, broadcast the lookup
good = (pixels.filter(F.col("crop_code") == 1).select("crop_code")
        .join(F.broadcast(names), "crop_code")
        .withColumn("area_ha", F.lit(PIXEL_HA))
        .groupBy("Class_Name").agg(F.sum("area_ha")))

bad.explain(mode="formatted")
good.explain(mode="formatted")
time_it("bad", lambda: bad.collect())
time_it("good", lambda: good.collect())

== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Initial Plan ==
   PhotonResultStage (17)
   +- PhotonColumnarToRow (16)
      +- PhotonGroupingAgg (15)
         +- PhotonShuffleExchangeSource (14)
            +- PhotonShuffleMapStage (13)
               +- PhotonShuffleExchangeSink (12)
                  +- PhotonGroupingAgg (11)
                     +- PhotonProject (10)
                        +- PhotonBroadcastHashJoin Inner (9)
                           :- PhotonScan parquet  (1)
                           +- PhotonShuffleExchangeSource (8)
                              +- PhotonShuffleMapStage (7)
                                 +- PhotonShuffleExchangeSink (6)
                                    +- PhotonProject (5)
                                       +- PhotonFilter (4)
                                          +- PhotonRowToColumnar (3)
                                             +- Scan csv  (2)


(1) PhotonScan parquet 
Output [1]: [crop_code#10234]
DictionaryFilters:

[Row(Class_Name='Corn', sum(area_ha)=39285561.60129416)]

In [0]:
agg = enriched.groupBy("Class_Name").agg(F.sum("area_ha").alias("ha"))

time_it("no cache – action 1", lambda: agg.count())
time_it("no cache – action 2", lambda: agg.collect())

try:
    agg.cache(); agg.count()                      # fills the cache
    time_it("cached – action 1", lambda: agg.count())
    time_it("cached – action 2", lambda: agg.collect())
except Exception as e:
    # Serverless compute may reject .cache(); materialize to Parquet instead
    print("cache() not supported here:", type(e).__name__)
    agg.write.mode("overwrite").parquet(f"{OUT}/_tmp_agg")
    mat = spark.read.parquet(f"{OUT}/_tmp_agg")
    time_it("materialized – action 1", lambda: mat.count())
    time_it("materialized – action 2", lambda: mat.collect())

no cache – action 1: 1.96s
no cache – action 2: 1.99s
cached – action 1: 0.50s
cached – action 2: 0.96s


In [0]:
t = time.perf_counter()
lazy_df = (pixels
    .filter(F.col("crop_code") == 1)
    .withColumn("area_ha", F.lit(PIXEL_HA))
    .groupBy((F.col("y") / 100000).cast("int").alias("y_band"))
    .agg(F.sum("area_ha").alias("corn_ha")))
print(f"Transformations defined in {time.perf_counter() - t:.4f}s: nothing executed yet (lazy)")

t = time.perf_counter()
rows = lazy_df.collect()
print(f"Action collect() ran the job in {time.perf_counter() - t:.2f}s and returned {len(rows)} rows (eager)")

Transformations defined in 0.0010s: nothing executed yet (lazy)
Action collect() ran the job in 1.63s and returned 29 rows (eager)


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

groups = ["grain", "oilseed", "hay_forage", "double_crop", "specialty_other"]
cell_mix = (enriched.groupBy("grid_x", "grid_y")
            .pivot("land_group", groups).agg(F.sum("area_ha")).na.fill(0)
            .withColumn("total_ha", sum(F.col(g) for g in groups))
            .filter("total_ha >= 10000"))
for g in groups:
    cell_mix = cell_mix.withColumn(f"{g}_share", F.col(g) / F.col("total_ha"))

features = VectorAssembler(inputCols=[f"{g}_share" for g in groups],
                           outputCol="features").transform(cell_mix)
model = KMeans(k=5, seed=42).fit(features)
clustered = model.transform(features)

display(clustered.groupBy("prediction")
        .agg(F.count("*").alias("cells"),
             *[F.round(F.avg(f"{g}_share"), 2).alias(g) for g in groups])
        .orderBy("prediction"))

prediction,cells,grain,oilseed,hay_forage,double_crop,specialty_other
0,315,0.33,0.45,0.08,0.05,0.09
1,360,0.78,0.04,0.07,0.02,0.09
2,126,0.17,0.06,0.07,0.03,0.67
3,522,0.53,0.33,0.08,0.03,0.03
4,233,0.37,0.05,0.48,0.01,0.08
